In [ ]:
# 1. Import the Google Earth Engine libraries.
#    Authenticate to Google Earth Engine. This
#    step only needs to be done once.

import ee

ee.Authenticate()

In [1]:
# 2. Initialize Google Earth Engine, using the 
#    high-volume endpoint. The high-volume
#    endpoint handles high request volumes better.
import ee

ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

In [2]:
# 2. Define a small Area of Interest (AOI).
#    Here we define a small bounding box in California 
#    for demonstration.

aoi = ee.Geometry.Polygon(
    [[[-120.5, 38.5],
      [-120.5, 38.55],
      [-120.4, 38.55],
      [-120.4, 38.5],
      [-120.5, 38.5]]]
)

In [3]:
# 3. Create the Google Earth Engine Image Collection.
#    Landsat 8, 5-year span, filtering to just the NIR 
#    band (SR_B5).

start_date = '2015-01-01'
end_date = '2019-12-31'

def prep_sr_l8(image):
    # Apply the scaling factors to the optical bands
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)

    # Replace the original bands with the scaled ones and apply the masks.
    return (image.addBands(optical_bands, None, True))

# Get Landsat 8, apply prep_sr_l8 function and select only NIR band.
ic = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
      .filterBounds(aoi)
      .filterDate(start_date, end_date)
      .map(prep_sr_l8)
      .select(['SR_B5']))

In [4]:
# 4. Write the custom user-defined function (UDF).

# Import necessary packages for our UDF.
import pandas as pd
import numpy as np

def compute_regression(df):
    """
    Computes linear regression (slope m, intercept b) of NIR (SR_B5) over time.
    The output must return a DataFrame with the same number of rows as the input chunk
    for robustraster's formatting to properly map it back to xarray.
    """
    df = df.copy()
    
    # Drop rows with NaNs in time or SR_B5 to safely calculate means
    valid_mask = df['SR_B5'].notna() & df['time'].notna()
    df_valid = df[valid_mask].copy()

    # Convert time to numeric years for interpretation of 'm' (change per year)
    df_valid['time_years'] = pd.to_datetime(df_valid['time']).astype('int64') // 10**9 / (3600 * 24 * 365.25)
    
    # We group by X and Y to calculate covariance and variance for linear regression:
    # m = Cov(x,y) / Var(x)
    # b = mean(y) - m * mean(x)
    grouped = df_valid.groupby(['X', 'Y'])
    
    df_valid['x_mean'] = grouped['time_years'].transform('mean')
    df_valid['y_mean'] = grouped['SR_B5'].transform('mean')
    
    df_valid['dx'] = df_valid['time_years'] - df_valid['x_mean']
    df_valid['dy'] = df_valid['SR_B5'] - df_valid['y_mean']
    
    df_valid['dx_dy'] = df_valid['dx'] * df_valid['dy']
    df_valid['dx_sq'] = df_valid['dx'] ** 2
    
    cov = grouped['dx_dy'].transform('sum')
    var = grouped['dx_sq'].transform('sum')
    
    # Calculate slope (m) and intercept (b)
    # Avoid division by zero by replacing 0 variance with nan
    df_valid['m'] = cov / var.replace(0, np.nan)
    df_valid['b'] = df_valid['y_mean'] - df_valid['m'] * df_valid['x_mean']
    
    # Merge results back to the original full dataframe shape
    df['m'] = np.nan
    df['b'] = np.nan
    df.loc[valid_mask, 'm'] = df_valid['m']
    df.loc[valid_mask, 'b'] = df_valid['b']
    
    # For any pixels with missing m and b, we can fill them with the group mean (which is the same value)
    df['m'] = df.groupby(['X', 'Y'])['m'].transform('first')
    df['b'] = df.groupby(['X', 'Y'])['b'].transform('first')
    
    # Output must return the target columns
    return df[['m', 'b']]


In [ ]:
# 5. Do the full computation

# Import the run() function from robustraster package.
from robustraster import run

print("Starting regression computation over time...")
    
# We want to chunk 'time' entirely (-1) so that all time steps for a pixel are loaded into memory together.
# X and Y chunk sizes determine the spatial tiling. 256 is usually a safe default for time-series stacks.
chunks = {"time": -1, "X": 256, "Y": 256}

run(
    dataset=ic,
    dataset_config={
        'geometry': aoi,
        'crs': 'EPSG:3310',
        'scale': 30,
    },
    user_function_config={
        "user_function": compute_regression,
    },
    function_tuning_config={
        "chunks": chunks,
    },
    export_config={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "Regression_Output",
        "vrt": True,
        "report": True
    },
)